# LSTM 时间序列预测

用PyTorch实现一个简单的LSTM来做月度销售预测。

说实话，36个月的训练数据对LSTM来说太少了，这更多是一个pipeline的练习。在实际项目中如果要用深度学习，至少需要几百个数据点，或者用daily数据。

**方法：**
- 用MinMaxScaler归一化到[0,1]
- 滑动窗口: 过去12个月 → 预测下1个月
- 测试时用递归预测（把自己的预测作为下一步的输入）

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw_data, get_monthly_sales, train_test_split_ts
from src.models.lstm_model import LSTMForecaster
from src.evaluate import evaluate_forecast, compare_models
from src.utils import plot_forecast, set_seed

set_seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
df = load_raw_data('../data/raw/train.csv')
monthly = get_monthly_sales(df)
train, test = train_test_split_ts(monthly, test_year=2018)
print(f'Train: {len(train)} months, Test: {len(test)} months')
print(f'训练样本 (扣除seq_len=12后): {len(train) - 12} 个滑动窗口')
print('...确实很少')

## 1. 基础LSTM

先用默认参数跑一版，2层LSTM + 64个hidden units。

In [ ]:
lstm = LSTMForecaster(
    seq_len=12,
    hidden_size=64,
    num_layers=2,
    lr=0.001,
    epochs=150,
    batch_size=8,
)
lstm.fit(train)

In [ ]:
fig = lstm.plot_loss()
plt.show()

loss收敛了但有些抖动，batch size太小数据量也少，正常。

In [ ]:
lstm_pred = lstm.predict(steps=len(test), test_index=test.index)

result = evaluate_forecast(test, lstm_pred, 'LSTM')
print(f"RMSE={result['RMSE']:.2f}, MAE={result['MAE']:.2f}, MAPE={result['MAPE']:.2f}%")

In [ ]:
fig = plot_forecast(train, test, lstm_pred, 'LSTM Forecast (seq=12, hidden=64)')
plt.savefig('../results/figures/lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 超参数对比

试几组不同的配置，看看有没有改善空间。

没做正式的grid search，数据量这么小没太大意义，手动试几个就好。

In [ ]:
configs = [
    {'seq_len': 6, 'hidden_size': 32, 'epochs': 150},
    {'seq_len': 12, 'hidden_size': 64, 'epochs': 150},  # same as above, for reference
    {'seq_len': 12, 'hidden_size': 128, 'epochs': 200},
]

lstm_results = []
for i, cfg in enumerate(configs):
    print(f'\n--- Config {i+1}: seq={cfg["seq_len"]}, hidden={cfg["hidden_size"]} ---')
    set_seed(42)
    model = LSTMForecaster(
        seq_len=cfg['seq_len'],
        hidden_size=cfg['hidden_size'],
        num_layers=2,
        lr=0.001,
        epochs=cfg['epochs'],
        batch_size=8,
    )
    model.fit(train)
    pred = model.predict(steps=len(test), test_index=test.index)
    name = f"LSTM(seq={cfg['seq_len']}, h={cfg['hidden_size']})"
    res = evaluate_forecast(test, pred, name)
    lstm_results.append(res)
    print(f'  RMSE={res["RMSE"]:.2f}')

In [ ]:
lstm_comparison = compare_models(lstm_results)
print(lstm_comparison.to_string())

加大模型容量（hidden=128）并没有带来明显提升，反而可能过拟合了。这进一步说明瓶颈在数据量而不是模型复杂度。

## 3. 和统计模型对比

In [ ]:
import os
if os.path.exists('../results/model_comparison.csv'):
    prev = pd.read_csv('../results/model_comparison.csv')
    print('All models comparison:')
    print(prev.to_string(index=False))
else:
    print('Run `python run_experiment.py` first to generate full comparison.')

## 总结

LSTM在这个任务上不如统计模型，原因分析：

1. **数据量不够** — 36个训练点，减去seq_len=12后只有24个滑动窗口样本，LSTM根本学不到足够的pattern
2. **递归预测误差累积** — 预测第2个月时用的是第1个月的预测值（不是真实值），误差会逐步放大。可以考虑Direct Multi-Output但需要更多数据
3. **季节性太强** — 当信号的主要成分是一个固定的12周期pattern时，Seasonal Naive这种直接copy的方法反而最准

如果要让LSTM有竞争力，可以尝试：
- 用日级数据（~1400个点）
- 加入外部特征（holiday indicator, promotion flag等）
- 按品类分别建模然后ensemble
- 用Transformer-based模型（但数据量问题依然存在）

但对当前这个小数据集来说，SARIMA或Prophet已经足够了。